In [5]:
import pandas as pd

from docplex.mp.model import Model

mdl = Model(name="crop_production")


In [7]:
df = pd.read_csv("crop_production_data.csv", encoding="utf-8-sig")
df

,crop_id,crop_name,profit_p_j,fixed_fee_F_j,min_bound_K_j,max_bound_L_j,req_land,req_capital,req_water
0,1,Rice,350,2000,15,150,1,200,5.0
1,2,Sweet corn,280,1500,10,120,1,140,2.5
2,3,Coconut,420,3500,8,70,1,250,1.5
3,4,Sugarcane,510,5000,20,100,1,450,4.5
4,5,Saba banana,480,4000,12,90,1,320,3.0


In [9]:
df_res = pd.read_csv("resource_limits_data.csv", encoding="utf-8-sig")
df_res

,resource_id,resource_name,limit_R_i
0,1,Arable Land (acres),300
1,2,Operating Capital (_),60000
2,3,Water Allocation (acre-feet),800


In [11]:
crops = df['crop_id'].tolist()
resources = df_res['resource_id'].tolist()

In [12]:
p = dict(zip(df["crop_id"], df["profit_p_j"]))                                                                       
F = dict(zip(df["crop_id"], df["fixed_fee_F_j"]))                                                                    
K = dict(zip(df["crop_id"], df["min_bound_K_j"]))                                                                    
L = dict(zip(df["crop_id"], df["max_bound_L_j"]))            
R = dict(zip(df_res["resource_id"], df_res["limit_R_i"])) 

In [13]:
a = {}                                               
for _, row in df.iterrows():                                                                                         
    j = int(row["crop_id"])                                                                                          
    a[(1, j)] = row["req_land"]     # Resource 1: Arable Land                                                        
    a[(2, j)] = row["req_capital"]  # Resource 2: Operating Capital                                                  
    a[(3, j)] = row["req_water"]    # Resource 3: Water Allocation

In [15]:
x = mdl.continuous_var_dict(crops, lb=0, name="hectares_planted")
y = mdl.binary_var_dict(crops, name="is_selected")

In [16]:
#obj function

total_profit = mdl.sum(p[j] * x[j] for j in crops)
total_fixed_cost = mdl.sum(F[j] * y[j] for j in crops)                                                  
mdl.maximize(total_profit - total_fixed_cost)

In [17]:
mdl.add_constraints(
    (mdl.sum(a[i, j] * x[j] for j in crops)<= R[i]) for i in resources
)

[docplex.mp.LinearConstraint[](hectares_planted_1+hectares_planted_2+hectares_planted_3+hectares_planted_4+hectares_planted_5,LE,300),
 docplex.mp.LinearConstraint[](200hectares_planted_1+140hectares_planted_2+250hectares_planted_3+450hectares_planted_4+320hectares_planted_5,LE,60000),
 docplex.mp.LinearConstraint[](5hectares_planted_1+2.500hectares_planted_2+1.500hectares_planted_3+4.500hectares_planted_4+3hectares_planted_5,LE,800)]

In [18]:
for j in crops:                                                                                                      
    # Minimum bound constraint: x_j >= K_j * y_j                                                                     
    mdl.add_constraint(x[j] >= K[j] * y[j], ctname=f"min_bound_crop_{j}")                                            
                                                                                                                        
    # Maximum bound constraint: x_j <= L_j * y_j                                                                     
    mdl.add_constraint(x[j] <= L[j] * y[j], ctname=f"max_bound_crop_{j}") 

In [19]:
print("Zach is solving.....")
solution = mdl.solve(log_output=True)

Zach is solving.....
Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.02 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 0 rows and 10 columns.
Reduced MIP has 13 rows, 10 columns, and 35 nonzeros.
Reduced MIP has 5 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.02 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 13 rows, 10 columns, and 35 nonzeros.
Reduced MIP has 5 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.01 sec. (0.02 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    

In [21]:
if solution:
    print("\n=== OPTIMAL SOLUTION FOUND ===")
    print(f"Status: {mdl.get_solve_status()}")
    print(f"Total Profit (Z): ${solution.objective_value:,.2f}")
    
    crop_names = dict(zip(df["crop_id"], df["crop_name"]))
    
    selected_crops = []
    hectares = []
    revenues = []
    costs = []
    
    for j in crops:
        # Check if crop j was selected
        if solution.get_value(y[j]) > 0.5:
            qty = solution.get_value(x[j])
            
            selected_crops.append(crop_names[j])
            hectares.append(qty)
            revenues.append(qty * p[j])
            costs.append(F[j])
            
    results_df = pd.DataFrame({
        'Selected Crop': selected_crops,
        'Hectares Planted': hectares,
        'Projected Revenue': revenues,
        'Production Cost': costs
    })
    
    print(results_df)
    
else:
    print("No feasible solution found.")


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Profit (Z): $93,015.00
  Selected Crop  Hectares Planted  Projected Revenue  Production Cost
0          Rice              49.3            17255.0             2000
1    Sweet corn             120.0            33600.0             1500
2       Coconut              70.0            29400.0             3500
3   Saba banana              49.5            23760.0             4000
